In [2]:
import os
import pandas as pd
from glob import glob
# ! pip install chardet

In [4]:
def get_filetered_csv(csv_file):
    _df = pd.read_csv(csv_file)
    _df = _df.loc[:, ~_df.columns.str.contains('^Unnamed')]
    _df.reset_index(drop=True, inplace=True)

    # _df = _df[_df['발생지시도'] == '서울']
    # 이게 [발생년월일시분] or [발생년월일시, 발생분] 
    if '발생년월일시분' in _df.columns:
        # 2015-01-01 05:57 형시
        _df['year'] = _df['발생년월일시분'].str.split(' ').str[0].str.split('-').str[0]
        _df['month'] = _df['발생년월일시분'].str.split(' ').str[0].str.split('-').str[1]
        _df['day'] = _df['발생년월일시분'].str.split(' ').str[0].str.split('-').str[2]
        _df['hour'] = _df['발생년월일시분'].str.split(' ').str[1].str.split(':').str[0].str.zfill(2)
        # _df['minute'] = _df['발생년월일시분'].str.split(' ').str[1].str.split(':').str[1]  
    else:
        _df['year'] = _df['발생년월일시'].str.split(' ').str[0].str.split('-').str[0]
        _df['month'] = _df['발생년월일시'].str.split(' ').str[0].str.split('-').str[1]
        _df['day'] = _df['발생년월일시'].str.split(' ').str[0].str.split('-').str[2]
        _df['hour'] = _df['발생년월일시'].str.split(' ').str[1].str.split(':').str[0].str.zfill(2)
    
    _df = _df[['year', 'month','day', 'hour', '주야', '요일', '발생지시도', '발생지시군구', '사고유형_대분류', '경도', '위도']]
    _df["datetime"] = pd.to_datetime(
        _df["year"] + _df["month"] + _df["day"] + _df["hour"],
        format="%Y%m%d%H"
    )
    
    # _df["발생일시"] = pd.to_datetime(_df["발생년월일시"], format="%Y-%m-%d %H")
    # _df = _df.rename(columns={'경도' : 'lon', '위도': 'lat'})
    # 경도 나 위도가 NaN인 경우는 삭제
    _df = _df[~_df['경도'].isna()]
    _df = _df[~_df['위도'].isna()]
    return _df

root= '../files2/origin'
file_list = sorted(glob(f"{root}/*_data.csv"))
df_list = [get_filetered_csv(f) for f in file_list]

df_all = pd.concat(df_list, axis=0)
df_all.reset_index(drop=True, inplace=True)
df_all = df_all.rename(columns={
    '경도' : 'lon', 
    '위도': 'lat',
    "발생지시도":      "city",
    "발생지시군구":    "district",
    "사고유형_대분류":"accident_type",
    '요일' : 'day_of_week',
    '주야' : 'day_night',
})
# df_all = df_all.set_index("datetime").sort_index()
df_all

,year,month,day,hour,day_night,day_of_week,city,district,accident_type,lon,lat,datetime
0,2012,05,09,01,야간,수,경남,거창군,차대차,127.853191,35.580719,2012-05-09 01:00:00
1,2012,03,30,13,주간,금,경기,용인시,차대차,127.052936,37.317615,2012-03-30 13:00:00
2,2012,10,26,20,야간,금,경북,김천시,차대차,128.003224,36.190921,2012-10-26 20:00:00
3,2012,06,28,14,주간,목,경기,이천시,차대차,127.420426,37.238956,2012-06-28 14:00:00
4,2012,06,18,04,야간,월,전남,광양시,차대차,127.648393,34.964371,2012-06-18 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
45065,2023,12,31,03,야,일,경기,파주시,차량단독,126.719489,37.702733,2023-12-31 03:00:00
45066,2023,12,31,08,주,일,강원,속초시,차대차,128.592780,38.185021,2023-12-31 08:00:00
45067,2023,12,31,11,주,일,경북,성주군,차량단독,128.180706,35.906771,2023-12-31 11:00:00
45068,2023,12,31,21,야,일,충남,천안시,차대사람,127.168430,36.828348,2023-12-31 21:00:00


In [ ]:
df_all[df_all['district'] =='수원시'].to_csv('suwon_accidents.csv', index=False)

,year,month,day,hour,day_night,day_of_week,city,district,accident_type,lon,lat,datetime
118,2012,01,08,01,야간,일,경기,수원시,차대사람,127.022127,37.260400,2012-01-08 01:00:00
168,2012,01,16,12,주간,월,경기,수원시,차대차,126.994672,37.316821,2012-01-16 12:00:00
246,2012,01,02,22,야간,월,경기,수원시,차대사람,127.035563,37.254027,2012-01-02 22:00:00
298,2012,01,05,15,주간,목,경기,수원시,차량단독,127.016527,37.279009,2012-01-05 15:00:00
361,2012,01,10,19,야간,화,경기,수원시,차대사람,126.999262,37.268022,2012-01-10 19:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
44954,2023,12,11,03,야,월,경기,수원시,차대사람,127.021513,37.232662,2023-12-11 03:00:00
44976,2023,12,14,21,야,목,경기,수원시,차대사람,127.000970,37.306563,2023-12-14 21:00:00
45006,2023,12,20,21,야,수,경기,수원시,차대사람,127.047643,37.293131,2023-12-20 21:00:00
45014,2023,12,22,13,주,금,경기,수원시,차대사람,126.999064,37.265578,2023-12-22 13:00:00


In [101]:
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.GeoDataFrame(
    df_all,
    geometry=[Point(xy) for xy in zip(df_all.lon, df_all.lat)],
    crs="EPSG:4326"
)
gdf.head()

,year,month,day,hour,day_night,day_of_week,city,district,accident_type,lon,lat,datetime,geometry
0,2012,01,13,16,주간,화,서울,강서구,차량단독,126.821995,37.544155,2012-01-13 16:00:00,POINT (126.82199 37.54416)
1,2012,07,09,10,주간,월,서울,서초구,차대차,127.051443,37.451040,2012-07-09 10:00:00,POINT (127.05144 37.45104)
2,2012,01,01,01,야간,일,서울,은평구,차대사람,126.931877,37.612850,2012-01-01 01:00:00,POINT (126.93188 37.61285)
3,2012,08,28,04,야간,화,서울,서초구,차대차,127.051336,37.451815,2012-08-28 04:00:00,POINT (127.05134 37.45181)
4,2012,01,07,23,야간,일,서울,강남구,차대차,127.050853,37.500513,2012-01-07 23:00:00,POINT (127.05085 37.50051)


In [102]:
df_all.to_csv('../files2/final/seoul_accidents.csv', index=False)

---

### (2) 보행등 

In [19]:
import chardet
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
        print(result['encoding'])  # 인코딩 확인
    df = pd.read_csv(file_path, encoding=result['encoding'])
    df.to_csv(file_path)

csv_roof = '../files2/origin'
# detect_encoding(f"{csv_roof}/seoul_traffic_lights.csv")
df_light = pd.read_csv(f"{csv_roof}/seoul_traffic_lights.csv")
df_light = df_light.loc[:, ~df_light.columns.str.contains('^Unnamed')]
df_light = df_light[['자치구', '신호등종류', '위도', '경도', '주소']]
df_light = df_light.rename(columns={
    '자치구' : 'district',
    '신호등종류' : 'light_type',
    '위도' : 'lat',
    '경도' : 'lon',
    '주소' : 'address'
})
df_light.to_csv(f'{csv_roof}/seoul_traffic_lights.csv', index=False)
df_light

,district,light_type,lat,lon,address
0,강남구,보행등,37.498560,127.044475,강남구 역삼동 710 대
1,강남구,보행등,37.509105,127.033164,강남구 논현동 219-12 대
2,강남구,보행등,37.495683,127.085581,강남구 일원동 460천
3,강남구,보행등,37.495809,127.085987,강남구 일원동 460천
4,강남구,보행등,37.534857,127.034393,강남구 압구정동 484천
...,...,...,...,...,...
24133,중랑구,보행등,37.616689,127.079436,중랑구 묵동 649-4대
24134,중랑구,보행등,37.616689,127.079436,중랑구 묵동 649-4대
24135,중랑구,보행등,37.614251,127.091812,중랑구 신내동 640 차
24136,중랑구,보행등,37.616534,127.079397,중랑구 묵동 656-2도
